# 05 Governance Layer

**Project:** Black Hills Mining Landscape Digital Twin Phase I
**Territory:** He Sapa (Black Hills) Unceded Lakota Territory

## Purpose

This notebook makes the governance architecture visible and auditable:

1. Validates that every record carries complete IEEE 2890-2025 provenance
2. Documents CARE and OCAP alignment of the Phase I dataset
3. Prepares the governance scaffold for Phase II Tribal data

A governance layer that lives only in documentation is not a governance
layer. This notebook runs checks against the actual data and produces
a compliance report.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
import pandas as pd
import geopandas as gpd

from src.constants import TREATY_PROVENANCE, GOVERNANCE_REFS, OUTPUTS_DIR
from src.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings('ignore')
%matplotlib inline

mines = gpd.read_file(OUTPUTS_DIR/'he_sapa_mine_gazetteer.geojson')
print(f'Gazetteer records : {len(mines):,}')
print(f'Total columns     : {len(mines.columns)}')

Gazetteer records : 1,719
Total columns     : 27


In [2]:
# Print data sovereignty acknowledgement at the top of every notebook
print_data_acknowledgment()


BLACK HILLS MINING DIGITAL TWIN TREATY TERRITORY ACKNOWLEDGMENT

He Sapa (the Black Hills) is unceded Lakota territory.

The 1868 Fort Laramie Treaty guaranteed He Sapa and surrounding lands to the Lakota
and their allies in perpetuity. After Custer's 1874 expedition confirmed gold, 
the US Congress unilaterally took the Black Hills in 1877. The US Supreme
Court ruled this action as unconstitutional in United States v. Sioux Nation of
Indians, 448 U.S. 371 (1980). The Sioux Nations have declined
the offered compensation. The Black Hills are not for sale.

Every record in this system carries that context as a provenance field.

GOVERNANCE FRAMEWORKS:

OCAP®  : The Oceti Sakowin and allied Nations have Ownership, Control, Access,
  and Possession of data describing their territory and resources.
  Phase I uses public federal data only. Any results describing
  He Sapa should be shared with relevant Lakota governance bodies
  before external publication or distribution.
  Reference: http

## IEEE 2890-2025 Compliance Audit

In [ ]:
REQUIRED_FIELDS = list(TREATY_PROVENANCE.keys()) + [
    'prov_source', 'prov_source_url', 'ai_generated', 'human_verified',
]

print('IEEE 2890-2025 PROVENANCE COMPLIANCE AUDIT')
all_pass = True
for field in REQUIRED_FIELDS:
    if field not in mines.columns:
        print(f'  MISSING FIELD : {field}')
        all_pass = False
        continue
    n_bad = mines[field].isna().sum() + (mines[field].astype(str).str.strip() == '').sum()
    status = 'OK' if n_bad == 0 else f'INCOMPLETE ({n_bad} records)'
    print(f'  {status:<22}  {field}')
    if n_bad > 0:
        all_pass = False

print()
print(f'Overall: {"PASS" if all_pass else "FAIL: see above"}')

IEEE 2890-2025 PROVENANCE COMPLIANCE AUDIT
  OK                      treaty_territory
  OK                      treaty_status
  OK                      legal_citation
  OK                      tk_label
  OK                      tk_label_note
  OK                      care_collective_benefit
  OK                      visibility_level
  OK                      phase_ii_sensitivity
  OK                      tribal_review_required
  OK                      redistribution_allowed
  OK                      ieee_2890_compliant
  OK                      data_steward_phase1
  OK                      data_steward_note
  OK                      prov_source
  OK                      prov_source_url
  OK                      ai_generated
  OK                      human_verified

Overall: PASS


## Territorial Provenance Verification

In [4]:
print('TERRITORIAL PROVENANCE FIELD VALUES')
for field, expected in TREATY_PROVENANCE.items():
    if field not in mines.columns:
        print(f'  MISSING: {field}')
        continue
    unique_vals = mines[field].unique()
    status = 'OK' if len(unique_vals) == 1 and str(unique_vals[0]) == str(expected) else 'WARN'
    print(f'  {status}  {field}')

print()
if 'treaty_status' in mines.columns:
    print('TREATY STATUS:')
    print(mines['treaty_status'].iloc[0])
if 'legal_citation' in mines.columns:
    print('\nLEGAL CITATION:')
    print(mines['legal_citation'].iloc[0])

TERRITORIAL PROVENANCE FIELD VALUES
  OK  treaty_territory
  OK  treaty_status
  OK  legal_citation
  OK  tk_label
  OK  tk_label_note
  OK  care_collective_benefit
  OK  visibility_level
  OK  phase_ii_sensitivity
  OK  tribal_review_required
  OK  redistribution_allowed
  OK  ieee_2890_compliant
  OK  data_steward_phase1
  OK  data_steward_note

TREATY STATUS:
Unceded Lakota territory. The 1877 Congressional act taking He Sapa was ruled unconstitutional by the US Supreme Court in United States v. Sioux Nation of Indians (1980). The Treaty Nations have declined compensation, maintaining that the land was never legally transferred.

LEGAL CITATION:
United States v. Sioux Nation of Indians, 448 U.S. 371 (1980)


## CARE Principles Alignment

In [10]:
print('CARE PRINCIPLES ALIGNMENT')

care = {
    'C : Collective Benefit': {
        'status': 'Partial',
        'evidence': 'Lakota team members now have their first complete mine inventory of He Sapa.',
        'gap': 'Results should be formally presented to Lakota governance bodies.',
        'phase2': 'Tribal-governed analysis layers will center Lakota priorities.',
    },
    'A : Authority to Control': {
        'status': 'Architecture-ready',
        'evidence': 'tribal_review_required on every record. Phase II scaffold in place.',
        'gap': 'No Lakota governance body has formally authorized Phase I data use yet.',
        'phase2': 'Phase II requires formal data sharing agreement with Lakota Nations.',
    },
    'R : Responsibility': {
        'status': 'Strong',
        'evidence': 'Full provenance chain. AI extraction flagged. Human review queue documented.',
        'gap': 'Human review of AI-extracted records not yet complete.',
        'phase2': 'Every record transformation must remain auditable.',
    },
    'E : Ethics': {
        'status': 'Strong',
        'evidence': 'Territorial framing uses Lakota legal position, not federal framing.',
        'gap': 'No formal ethics review process for Phase II data intake yet.',
        'phase2': 'Ethics review must be co-designed with Lakota team members.',
    },
}

for principle, a in care.items():
    print(f'\n  {principle}')
    print(f'    Status  : {a["status"]}')
    print(f'    Evidence: {a["evidence"]}')
    print(f'    Gap     : {a["gap"]}')
    print(f'    Phase II: {a["phase2"]}')

CARE PRINCIPLES ALIGNMENT

  C : Collective Benefit
    Status  : Partial
    Evidence: Lakota team members now have their first complete mine inventory of He Sapa.
    Gap     : Results should be formally presented to Lakota governance bodies.
    Phase II: Tribal-governed analysis layers will center Lakota priorities.

  A : Authority to Control
    Status  : Architecture-ready
    Evidence: tribal_review_required on every record. Phase II scaffold in place.
    Gap     : No Lakota governance body has formally authorized Phase I data use yet.
    Phase II: Phase II requires formal data sharing agreement with Lakota Nations.

  R : Responsibility
    Status  : Strong
    Evidence: Full provenance chain. AI extraction flagged. Human review queue documented.
    Gap     : Human review of AI-extracted records not yet complete.
    Phase II: Every record transformation must remain auditable.

  E : Ethics
    Status  : Strong
    Evidence: Territorial framing uses Lakota legal position, n

## OCAP Assessment

In [12]:
print('OCAP ASSESSMENT')

ocap = [
    ('Ownership',  'Lakota Nations own data describing their territory. TREATY_PROVENANCE documents this on every record.'),
    ('Control',    'Phase I stewardship is temporary. ACTION REQUIRED: formal presentation to Lakota governance bodies.'),
    ('Access',     'All Phase I data in open formats. Lakota team members have full access. Broader community access to be designed with Tribal input.'),
    ('Possession', 'Phase I outputs on Daear Consulting, LLC infrastructure. Phase II must include transfer of data custody to Lakota-controlled infrastructure.'),
]

for principle, text in ocap:
    print(f'\n  {principle}:')
    print(f'    {text}')

OCAP ASSESSMENT

  Ownership:
    Lakota Nations own data describing their territory. TREATY_PROVENANCE documents this on every record.

  Control:
    Phase I stewardship is temporary. ACTION REQUIRED: formal presentation to Lakota governance bodies.

  Access:
    All Phase I data in open formats. Lakota team members have full access. Broader community access to be designed with Tribal input.

  Possession:
    Phase I outputs on Daear Consulting, LLC infrastructure. Phase II must include transfer of data custody to Lakota-controlled infrastructure.


## Local Contexts TK Notice

In [7]:
print('LOCAL CONTEXTS TK NOTICE')
print()
print('Applied to all records in this system.')
print()
tk_val = mines['tk_label'].iloc[0] if 'tk_label' in mines.columns else 'NOT SET'
tk_note = mines['tk_label_note'].iloc[0][:80] if 'tk_label_note' in mines.columns else 'NOT SET'
print(f'  tk_label      : {tk_val}')
print(f'  tk_label_note : {tk_note}...')
print()
print('Reference: https://localcontexts.org/label/tk-notice/')
print()
print('Phase II: Specific Lakota-authorized TK and BC labels will replace')
print('the generic TK Notice once Nations have reviewed and authorized them.')

LOCAL CONTEXTS TK NOTICE

Applied to all records in this system.

  tk_label      : TK Notice
  tk_label_note : Indigenous interests exist in all records describing He Sapa. Contact originatin...

Reference: https://localcontexts.org/label/tk-notice/

Phase II: Specific Lakota-authorized TK and BC labels will replace
the generic TK Notice once Nations have reviewed and authorized them.


## Phase II Readiness Checklist

In [8]:
print('PHASE II READINESS CHECKLIST')

checklist = [
    ('DONE', 'Treaty provenance on every Phase I record'),
    ('DONE', 'IEEE 2890-2025 fields on every record'),
    ('DONE', 'TK Notice label on every record'),
    ('DONE', 'AI extraction flagged (ai_generated field)'),
    ('DONE', 'Human review queue template created'),
    ('DONE', 'CARE/OCAP alignment documented'),
    ('DONE', 'visibility_level and phase_ii_sensitivity fields present'),
    ('TODO', 'Formal presentation of Phase I to Lakota governance bodies'),
    ('TODO', 'Data sharing agreement with Lakota Nations for Phase II'),
    ('TODO', 'Phase II data intake ethics review process'),
    ('TODO', 'Nation-specific TK/BC label authorization'),
    ('TODO', 'Transfer plan: data custody to Lakota-controlled infrastructure'),
    ('TODO', 'Restricted data access controls for Phase II'),
    ('TODO', 'Oral history intake protocol (Phase II)'),
]

for status, item in checklist:
    icon = chr(10003) if status == 'DONE' else 'o'
    print(f'  {icon}  [{status}]  {item}')

n_done = sum(1 for s, _ in checklist if s == 'DONE')
print(f'\n  {n_done} of {len(checklist)} items complete')
print(f'  {len(checklist)-n_done} items required before Phase II data can be accepted')

PHASE II READINESS CHECKLIST
  ✓  [DONE]  Treaty provenance on every Phase I record
  ✓  [DONE]  IEEE 2890-2025 fields on every record
  ✓  [DONE]  TK Notice label on every record
  ✓  [DONE]  AI extraction flagged (ai_generated field)
  ✓  [DONE]  Human review queue template created
  ✓  [DONE]  CARE/OCAP alignment documented
  ✓  [DONE]  visibility_level and phase_ii_sensitivity fields present
  o  [TODO]  Formal presentation of Phase I to Lakota governance bodies
  o  [TODO]  Data sharing agreement with Lakota Nations for Phase II
  o  [TODO]  Phase II data intake ethics review process
  o  [TODO]  Nation-specific TK/BC label authorization
  o  [TODO]  Transfer plan: data custody to Lakota-controlled infrastructure
  o  [TODO]  Restricted data access controls for Phase II
  o  [TODO]  Oral history intake protocol (Phase II)

  7 of 14 items complete
  7 items required before Phase II data can be accepted
